# **Estudo do Fluxo de Pacientes por Acidentes com Animais Peçonhentos no Estado de São Paulo**
Este notebook realiza uma análise dos fluxos municipais e intermunicipais de pacientes vítimas de acidentes com animais peçonhentos, com base nos dados do SINAN/DATASUS. O objetivo principal é fornecer subsídios técnicos para a alocação eficiente de unidades de atendimento com soroterapia no estado de São Paulo.

Os dados utilizados neste estudo foram extraídos do DATASUS em 09 de março de 2025, por meio do sistema SINAN (Sistema de Informação de Agravos de Notificação). O conjunto contempla registros de acidentes envolvendo animais peçonhentos ocorridos no estado de São Paulo, no período de 2007 a 2023.

## **1. Leitura dos dados**

In [60]:
import pandas as pd
import matplotlib.pyplot as plt

In [61]:
# Importando os dados
!gdown --id 1KooD2IWFeVRFLrEgn41d7IAhW7lyZAR5 --output arquivo.csv

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1KooD2IWFeVRFLrEgn41d7IAhW7lyZAR5
From (redirected): https://drive.google.com/uc?id=1KooD2IWFeVRFLrEgn41d7IAhW7lyZAR5&confirm=t&uuid=9430b798-0cf4-4478-bd33-b90d7c0eca09
To: /content/arquivo.csv
100% 156M/156M [00:03<00:00, 46.7MB/s]


In [62]:
# Carregar o CSV
df = pd.read_csv('arquivo.csv')
df.head()

,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ANO_NASC,NU_IDADE_N,CS_SEXO,CS_GESTANT,CS_RACA,...,lon_origem,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,cid_destino,lat_destino,lon_destino,distancia_km
0,2007-01-25,200704,2007,35,350280,1958.0,4048,M,6.0,1.0,...,-50.4401,Araçatuba,3531,RRAS12,35021,CENTRAL DO DRS II,aracatuba,-21.2076,-50.4401,0.000000
1,2007-01-29,200705,2007,35,354850,1960.0,4046,M,6.0,NaN,...,-46.2580,Santos,3520,RRAS7,35041,BAIXADA SANTISTA,santos,-23.9535,-46.3350,8.757794
2,2007-01-22,200704,2007,35,351660,NaN,4018,M,6.0,4.0,...,-49.6546,Gália,3533,RRAS10,35093,MARILIA,galia,-22.2918,-49.5504,13.873489
3,2007-01-25,200704,2007,35,351660,1967.0,4039,F,5.0,4.0,...,-49.5504,Gália,3533,RRAS10,35093,MARILIA,galia,-22.2918,-49.5504,0.000000
4,2007-01-03,200701,2007,35,351660,1988.0,4018,M,6.0,1.0,...,-49.0871,Gália,3533,RRAS10,35093,MARILIA,galia,-22.2918,-49.5504,47.875244


In [63]:
df.columns

Index(['DT_NOTIFIC', 'SEM_NOT', 'NU_ANO', 'SG_UF_NOT', 'ID_MUNICIP',
       'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA',
       'CS_ESCOL_N', 'ANT_DT_ACI', 'ANT_UF', 'ANT_MUNIC_', 'ANT_TEMPO_',
       'ANT_LOCA_1', 'TP_ACIDENT', 'ANI_TIPO_1', 'ANI_SERPEN', 'ANI_ARANHA',
       'ANI_LAGART', 'TRA_CLASSI', 'CON_SOROTE', 'NU_AMPOLAS', 'NU_AMPOL_1',
       'NU_AMPOL_8', 'NU_AMPOL_6', 'NU_AMPOL_4', 'NU_AMPO_7', 'NU_AMPO_5',
       'NU_AMPOL_9', 'NU_AMPOL_3', 'DOENCA_TRA', 'EVOLUCAO', 'nome_origem',
       'cod_macro_origem', 'macro_origem', 'cod_RS_origem', 'RS_origem',
       'cid_origem', 'lat_origem', 'lon_origem', 'nome_destino',
       'cod_macro_destino', 'macro_destino', 'cod_RS_destino', 'RS_destino',
       'cid_destino', 'lat_destino', 'lon_destino', 'distancia_km'],
      dtype='object')

## **2. Análise Exploratória dos Fluxos**

### **2.1 Funções auxiliares**

In [64]:
# Número total de acidentes
tam_df = len(df)
tam_df

537751

In [65]:
# Função auxiliar para calcular porcentagem em relação ao número total de acidentes
def get_percent(t):
    return round((t / tam_df) * 100, 3)

### **2.2 Análise dos fluxos**

In [66]:
# Separação entre intermunicipais e municipais
df_intermunicipal = df[df["ID_MUNICIP"] != df["ANT_MUNIC_"]]
df_municipal = df[df["ID_MUNICIP"] == df["ANT_MUNIC_"]]

In [67]:
# Número de acidentes com fluxo intermunicipal
tam_intermun = len(df_intermunicipal)
print(f"O número de atendimentos intermunicipais é: {tam_intermun}, representado em {get_percent(tam_intermun)}% do total de atendimentos")

O número de atendimentos intermunicipais é: 49679, representado em 9.238% do total de atendimentos


In [68]:
# Número de acidentes sem deslocamento
tam_municipal = len(df_municipal)
print(f"O número de atendimentos dentro dos municípios é: {tam_municipal}, representado em {get_percent(tam_municipal)}% do total de atendimentos")

O número de atendimentos dentro dos municípios é: 488072, representado em 90.762% do total de atendimentos


In [69]:
# Dados do fluxo intermunicipal
df_intermunicipal = df_intermunicipal.groupby(["ID_MUNICIP", "nome_destino", "cod_macro_destino", "macro_destino", "cod_RS_destino", "RS_destino", "lat_destino", "lon_destino", "ANT_MUNIC_", "nome_origem", "cod_macro_origem", "macro_origem", "cod_RS_origem",  "RS_origem", "lat_origem", "lon_origem", "distancia_km"]).size().reset_index(name="total_ocorrencias")
df_intermunicipal

,ID_MUNICIP,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,lat_destino,lon_destino,ANT_MUNIC_,nome_origem,cod_macro_origem,macro_origem,cod_RS_origem,RS_origem,lat_origem,lon_origem,distancia_km,total_ocorrencias
0,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,350890.0,Caiabu,3532,RRAS11,35112,ALTA SOROCABANA,-22.0127,-51.2394,40.425971,1
1,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,351580.0,Flora Rica,3532,RRAS11,35111,ALTA PAULISTA,-21.6727,-51.3821,31.934183,1
2,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,351600.0,Flórida Paulista,3533,RRAS10,35091,ADAMANTINA,-21.6127,-51.1724,12.777554,22
3,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352080.0,Inúbia Paulista,3533,RRAS10,35091,ADAMANTINA,-21.7695,-50.9633,14.977616,15
4,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352740.0,Lucélia,3533,RRAS10,35091,ADAMANTINA,-21.7182,-51.0215,6.726297,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4160,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355170.0,Sertãozinho,3530,RRAS13,35131,HORIZONTE VERDE,-21.1316,-47.9875,221.588471,1
4161,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355340.0,Tanabi,3531,RRAS12,35155,SAO JOSE DO RIO PRETO,-20.6228,-49.6563,40.153836,3
4162,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355610.0,Valentim Gentil,3531,RRAS12,35157,VOTUPORANGA,-20.4217,-50.0889,11.565752,32
4163,355720,Chavantes,3533,RRAS10,35094,OURINHOS,-23.0366,-49.7096,353470.0,Ourinhos,3533,RRAS10,35094,OURINHOS,-22.9797,-49.8697,17.580933,1


In [70]:
# Número total de acidentes com deslocamento
df_intermunicipal['total_ocorrencias'].sum()

np.int64(49679)

In [71]:
# Salvar arquivos csv dos fluxos
df_intermunicipal.to_csv("fluxo_intermun.csv", index=False, encoding="utf-8")
df_municipal.to_csv("fluxo_mun.csv", index=False, encoding="utf-8")

#### **2.2.1 Fluxo intermunicipal**

##### **2.2.1.1 Deslocamento entre Macrorregião de Saúde**

In [72]:
# Dados com Macrorregiões distintas para origem (acidente) e destino (notificação)
df_macrodiferente = df_intermunicipal[df_intermunicipal["cod_macro_destino"] != df_intermunicipal["cod_macro_origem"]]
df_macrodiferente

,ID_MUNICIP,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,lat_destino,lon_destino,ANT_MUNIC_,nome_origem,cod_macro_origem,macro_origem,cod_RS_origem,RS_origem,lat_origem,lon_origem,distancia_km,total_ocorrencias
0,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,350890.0,Caiabu,3532,RRAS11,35112,ALTA SOROCABANA,-22.0127,-51.2394,40.425971,1
1,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,351580.0,Flora Rica,3532,RRAS11,35111,ALTA PAULISTA,-21.6727,-51.3821,31.934183,1
6,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352920.0,Martinópolis,3532,RRAS11,35112,ALTA SOROCABANA,-22.1462,-51.1709,52.372420,1
9,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,353540.0,Panorama,3532,RRAS11,35111,ALTA PAULISTA,-21.3540,-51.8562,88.835348,1
17,350030,Aguaí,3528,RRAS15,35142,MANTIQUEIRA,-22.0572,-46.9735,353930.0,Pirassununga,3529,RRAS14,35101,ARARAS,-21.9960,-47.4257,47.175680,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119,355690,Vista Alegre do Alto,3530,RRAS13,35052,SUL - BARRETOS,-21.1692,-48.6284,354980.0,São José do Rio Preto,3531,RRAS12,35155,SAO JOSE DO RIO PRETO,-20.8113,-49.3758,87.231781,1
4122,355690,Vista Alegre do Alto,3530,RRAS13,35052,SUL - BARRETOS,-21.1692,-48.6284,355370.0,Taquaritinga,3535,RRAS 18,35033,NOROESTE DO DRS III,-21.4049,-48.5103,28.831216,1
4155,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354580.0,Santa Bárbara d'Oeste,3528,RRAS15,35072,REGIAO METROPOLITANA DE CAMPINAS,-22.7553,-47.4143,370.301812,1
4156,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354890.0,São Carlos,3535,RRAS 18,35032,CORACAO DO DRS III,-22.0174,-47.8860,279.830086,1


In [73]:
# Número de ocorrências com deslocamento entre Macrorregiões
df_macrodiferente['total_ocorrencias'].sum()

np.int64(7889)

In [74]:
# Número de ocorrências com deslocamento entre Macrorregiões e Regionais de Sáude (RS)
df_macrodiferente_RSdiferente = df_macrodiferente[df_macrodiferente["cod_RS_destino"] != df_macrodiferente["cod_RS_origem"]]
df_macrodiferente_RSdiferente

,ID_MUNICIP,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,lat_destino,lon_destino,ANT_MUNIC_,nome_origem,cod_macro_origem,macro_origem,cod_RS_origem,RS_origem,lat_origem,lon_origem,distancia_km,total_ocorrencias
0,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,350890.0,Caiabu,3532,RRAS11,35112,ALTA SOROCABANA,-22.0127,-51.2394,40.425971,1
1,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,351580.0,Flora Rica,3532,RRAS11,35111,ALTA PAULISTA,-21.6727,-51.3821,31.934183,1
6,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352920.0,Martinópolis,3532,RRAS11,35112,ALTA SOROCABANA,-22.1462,-51.1709,52.372420,1
9,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,353540.0,Panorama,3532,RRAS11,35111,ALTA PAULISTA,-21.3540,-51.8562,88.835348,1
17,350030,Aguaí,3528,RRAS15,35142,MANTIQUEIRA,-22.0572,-46.9735,353930.0,Pirassununga,3529,RRAS14,35101,ARARAS,-21.9960,-47.4257,47.175680,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119,355690,Vista Alegre do Alto,3530,RRAS13,35052,SUL - BARRETOS,-21.1692,-48.6284,354980.0,São José do Rio Preto,3531,RRAS12,35155,SAO JOSE DO RIO PRETO,-20.8113,-49.3758,87.231781,1
4122,355690,Vista Alegre do Alto,3530,RRAS13,35052,SUL - BARRETOS,-21.1692,-48.6284,355370.0,Taquaritinga,3535,RRAS 18,35033,NOROESTE DO DRS III,-21.4049,-48.5103,28.831216,1
4155,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354580.0,Santa Bárbara d'Oeste,3528,RRAS15,35072,REGIAO METROPOLITANA DE CAMPINAS,-22.7553,-47.4143,370.301812,1
4156,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354890.0,São Carlos,3535,RRAS 18,35032,CORACAO DO DRS III,-22.0174,-47.8860,279.830086,1


In [75]:
# Número de ocorrências com deslocamento entre Macrorregiões e RS
tam_Mdiferente = df_macrodiferente_RSdiferente['total_ocorrencias'].sum()
tam_Mdiferente

np.int64(7889)

Portanto, todos com macroregiões (macro) diferentes, consequentemente, estão em regionais de saúde (RS) distintas, considerando origem (local do acidente) e destino (local da notificação/atendimento).

In [76]:
df_macrodiferente.to_csv("fluxo_intermun_Macrodiferente.csv", index=False, encoding="utf-8")

##### **2.2.1.2 Deslocamento entre Regionais de Saúde**

In [77]:
# Dados com deslocamento intermunicipal dentro da mesma macrorregião
df_macroigual = df_intermunicipal[df_intermunicipal["cod_macro_destino"] == df_intermunicipal["cod_macro_origem"]]
df_macroigual

,ID_MUNICIP,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,lat_destino,lon_destino,ANT_MUNIC_,nome_origem,cod_macro_origem,macro_origem,cod_RS_origem,RS_origem,lat_origem,lon_origem,distancia_km,total_ocorrencias
2,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,351600.0,Flórida Paulista,3533,RRAS10,35091,ADAMANTINA,-21.6127,-51.1724,12.777554,22
3,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352080.0,Inúbia Paulista,3533,RRAS10,35091,ADAMANTINA,-21.7695,-50.9633,14.977616,15
4,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352740.0,Lucélia,3533,RRAS10,35091,ADAMANTINA,-21.7182,-51.0215,6.726297,35
5,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352890.0,Mariápolis,3533,RRAS10,35091,ADAMANTINA,-21.7959,-51.1824,16.896965,17
7,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,353460.0,Osvaldo Cruz,3533,RRAS10,35091,ADAMANTINA,-21.7968,-50.8793,23.791128,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4159,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355130.0,Sebastianópolis do Sul,3531,RRAS12,35157,VOTUPORANGA,-20.6523,-49.9250,25.907300,28
4161,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355340.0,Tanabi,3531,RRAS12,35155,SAO JOSE DO RIO PRETO,-20.6228,-49.6563,40.153836,3
4162,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355610.0,Valentim Gentil,3531,RRAS12,35157,VOTUPORANGA,-20.4217,-50.0889,11.565752,32
4163,355720,Chavantes,3533,RRAS10,35094,OURINHOS,-23.0366,-49.7096,353470.0,Ourinhos,3533,RRAS10,35094,OURINHOS,-22.9797,-49.8697,17.580933,1


In [78]:
# Número de ocorrência intermunicipal na mesma macrorregião
tam_Migual = df_macroigual['total_ocorrencias'].sum()
tam_Migual

np.int64(41790)

In [79]:
df_macroigual.to_csv("fluxo_intermun_Migual.csv", index=False, encoding="utf-8")

In [80]:
# Dados intermunicipais na mesma macrorregião com RS diferente
df_RSdiferente = df_macroigual[df_macroigual["cod_RS_destino"] != df_macroigual["cod_RS_origem"]]
df_RSdiferente

,ID_MUNICIP,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,lat_destino,lon_destino,ANT_MUNIC_,nome_origem,cod_macro_origem,macro_origem,cod_RS_origem,RS_origem,lat_origem,lon_origem,distancia_km,total_ocorrencias
15,350030,Aguaí,3528,RRAS15,35142,MANTIQUEIRA,-22.0572,-46.9735,351080.0,Casa Branca,3528,RRAS15,35143,RIO PARDO,-21.7708,-47.0852,33.747661,7
16,350030,Aguaí,3528,RRAS15,35142,MANTIQUEIRA,-22.0572,-46.9735,353070.0,Mogi Guaçu,3528,RRAS15,35141,BAIXA MOGIANA,-22.3675,-46.9428,34.505992,3
18,350030,Aguaí,3528,RRAS15,35142,MANTIQUEIRA,-22.0572,-46.9735,354800.0,Santo Antônio de Posse,3528,RRAS15,35072,REGIAO METROPOLITANA DE CAMPINAS,-22.6029,-46.9192,60.686336,1
22,350050,Águas de Lindóia,3528,RRAS15,35074,CIRCUITO DAS AGUAS,-22.4733,-46.6314,350030.0,Aguaí,3528,RRAS15,35142,MANTIQUEIRA,-22.0572,-46.9735,58.019685,1
24,350050,Águas de Lindóia,3528,RRAS15,35074,CIRCUITO DAS AGUAS,-22.4733,-46.6314,352260.0,Itapira,3528,RRAS15,35141,BAIXA MOGIANA,-22.4357,-46.8224,20.095679,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4151,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,353730.0,Penápolis,3531,RRAS12,35023,CONSORCIOS DO DRS II,-21.4148,-50.0769,110.210888,1
4152,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,353990.0,Poloni,3531,RRAS12,35156,JOSE BONIFACIO,-20.7829,-49.8258,42.819652,1
4157,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354925.0,São João de Iracema,3531,RRAS12,35154,FERNANDOPOLIS,-20.5111,-50.3561,40.608166,1
4158,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354980.0,São José do Rio Preto,3531,RRAS12,35155,SAO JOSE DO RIO PRETO,-20.8113,-49.3758,76.043952,2


In [81]:
# Número de ocorrência intermunicipal na mesma macrorregião com RS diferente
tam_RSdiferente = df_RSdiferente['total_ocorrencias'].sum()
tam_RSdiferente

np.int64(4094)

In [82]:
df_RSdiferente.to_csv("fluxo_intermun_RSdiferente.csv", index=False, encoding="utf-8")

In [83]:
# Dados intermunicipais na mesma macrorregião com RS igual
df_RSigual = df_macroigual[df_macroigual["cod_RS_destino"] == df_macroigual["cod_RS_origem"]]
df_RSigual

,ID_MUNICIP,nome_destino,cod_macro_destino,macro_destino,cod_RS_destino,RS_destino,lat_destino,lon_destino,ANT_MUNIC_,nome_origem,cod_macro_origem,macro_origem,cod_RS_origem,RS_origem,lat_origem,lon_origem,distancia_km,total_ocorrencias
2,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,351600.0,Flórida Paulista,3533,RRAS10,35091,ADAMANTINA,-21.6127,-51.1724,12.777554,22
3,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352080.0,Inúbia Paulista,3533,RRAS10,35091,ADAMANTINA,-21.7695,-50.9633,14.977616,15
4,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352740.0,Lucélia,3533,RRAS10,35091,ADAMANTINA,-21.7182,-51.0215,6.726297,35
5,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,352890.0,Mariápolis,3533,RRAS10,35091,ADAMANTINA,-21.7959,-51.1824,16.896965,17
7,350010,Adamantina,3533,RRAS10,35091,ADAMANTINA,-21.6820,-51.0737,353460.0,Osvaldo Cruz,3533,RRAS10,35091,ADAMANTINA,-21.7968,-50.8793,23.791128,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4154,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,354420.0,Riolândia,3531,RRAS12,35157,VOTUPORANGA,-19.9868,-49.6836,57.330199,13
4159,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355130.0,Sebastianópolis do Sul,3531,RRAS12,35157,VOTUPORANGA,-20.6523,-49.9250,25.907300,28
4162,355710,Votuporanga,3531,RRAS12,35157,VOTUPORANGA,-20.4237,-49.9781,355610.0,Valentim Gentil,3531,RRAS12,35157,VOTUPORANGA,-20.4217,-50.0889,11.565752,32
4163,355720,Chavantes,3533,RRAS10,35094,OURINHOS,-23.0366,-49.7096,353470.0,Ourinhos,3533,RRAS10,35094,OURINHOS,-22.9797,-49.8697,17.580933,1


In [84]:
# Número de ocorrência intermunicipal na mesma macrorregião e RS
tam_RSigual = df_RSigual['total_ocorrencias'].sum()
tam_RSigual

np.int64(37696)

In [85]:
df_RSigual.to_csv("fluxo_intermun_RSigual.csv", index=False, encoding="utf-8")

### **2.3 Tabelas**

In [86]:
# Função auxiliar para gerar tabelas
def criar_tabela(df, titulo, nome_arquivo_csv, nome_arquivo_imagem):
    # Salvar CSV
    df.to_csv(nome_arquivo_csv, index=False, encoding="utf-8")

    # Criar figura com altura adaptável ao número de linhas
    fig, ax = plt.subplots(figsize=(10, 2 + 0.4 * len(df)))
    ax.set_axis_off()

    # Criar a tabela
    tabela = ax.table(cellText=df.values,
                      colLabels=df.columns,
                      cellLoc='center',
                      loc='center')

    # Estilo da tabela
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(9)
    tabela.auto_set_column_width(list(range(len(df.columns))))

    for key, cell in tabela.get_celld().items():
        cell.set_edgecolor("black")
        cell.set_linewidth(1.0)
        if key[0] == 0:  # Cabeçalho
            cell.set_fontsize(11)
            cell.set_text_props(weight='bold')
            cell.set_facecolor("white")
        else:
            cell.set_facecolor("white")

    # Título
    plt.title(titulo, fontsize=14, fontweight='bold')

    # Salvar imagem
    plt.savefig(nome_arquivo_imagem, dpi=300, bbox_inches='tight')
    plt.close(fig)

    return nome_arquivo_csv, nome_arquivo_imagem

In [87]:
# Cálculo da distância média de deslocamento
def distancia_media(df):
    if "distancia_km" in df.columns and "total_ocorrencias" in df.columns:
        total_ocorrencias = df["total_ocorrencias"].sum()
        if total_ocorrencias > 0:
            return (df["distancia_km"] * df["total_ocorrencias"]).sum() / total_ocorrencias
    return 0

#### **2.3.1 Tabela geral de fluxo**

In [88]:
# Tabela: Tabela Geral (Municipal e Intermunicipal)
resumo_geral_df = pd.DataFrame({
    "Tipo de Ocorrência": ["Municipal", "Intermunicipal"],
    "Número de Ocorrências": [tam_municipal, tam_intermun],
    "Percentual (%)": [get_percent(tam_municipal),
        get_percent(tam_intermun)]
})

csv_geral, img_geral = criar_tabela(resumo_geral_df, "Distribuição Geral dos Atendimentos por Tipo de Ocorrência",
                                    "resumo_fluxos_geral.csv", "tabela_fluxos_geral.png")

#### **2.3.2 Tabela de Macrorregião**

In [89]:
# Tabela: Intermunicipal por Macrorregião (M)
resumo_m_df = pd.DataFrame({
    "Fluxo": ["Mesma macrorregião", "Macrorregião distinta"],
    "Número de Ocorrências": [tam_Migual, tam_Mdiferente],
    "Percentual em relação ao total de acidentes (%)": [get_percent(tam_Migual), get_percent(tam_Mdiferente)],
    "Distância Média (km)": [distancia_media(df_macroigual).round(2), distancia_media(df_macrodiferente).round(2)]
})

csv_m, img_m = criar_tabela(resumo_m_df, "Análise dos Fluxos Intermunicipais entre Macrorregiões de Saúde",
                            "resumo_fluxos_M.csv", "tabela_fluxos_M.png")

#### **2.3.3 Tabela de RS**

In [90]:
# Tabela: Intermunicipal por Região de Saúde (RS)
resumo_rs_df = pd.DataFrame({
    "Tipo de Fluxo": ["Mesma regional de saúde", "Regional de saúde distinta"],
    "Número de Ocorrências": [tam_RSigual, tam_RSdiferente],
    "Percentual em relação ao total de acidentes (%)": [get_percent(tam_RSigual), get_percent(tam_RSdiferente)],
    "Distância Média (km)": [distancia_media(df_RSigual).round(2), distancia_media(df_RSdiferente).round(2)]
})


csv_rs, img_rs = criar_tabela(resumo_rs_df, "Análise dos Fluxos Intermunicipais entre Regionais de Saúde",
                              "resumo_fluxos_RS.csv", "tabela_fluxos_RS.png")